# Today in Day 4 and it is to do with, A Real Document Knowledge Graph, Because " What Users Asked About" isn't the Same Graph as "What The Document Says"

## A Distinction worth being precise about before writing any code 
Days 1-3 built a real, working graph and it's worth naming exactly what kind. `(User-[:ASKED_ABOUT]->(Entity)-[:RAISED_BY_QUESTION]->(Question))` is a graph of **user interactions**. It's a genuine, structural upgrade over Month 2's flat MonngoDB entity memory, and it correctly fixed the over-merging bug. But it has never once contained a single fact from the Apple 10-K. Ask it "how does Apple's Services segment revenue growth compare to its iPone segment revenue growth from FY2023 to FY2024" and there's nothing to traverse, that question is about the document's content, not about what any user has previously asked.

This is exactly the kind of question a vector store structurally struggles with, for the same reason Month 3 Day 5 had to build query decomposition: one similarity search against one embedded question has to somehow surface four separate numbers (Services FY23, Services FY24, iPhoneFY23, iPhoneFY24) living in different parts of the document, and there's no guarantee all four land in the same top-k retrieval. A **document knowledge graph** nodes for segments and financial facts, connected by real, extracted relationships, offers a genuinely different fix: if the facts are already sitting in the graph, one Cypher query pulls all four deterministically. Worth being honest about the real trade-off up front: This only works for facts that were extracted *ahead of time*. The vector store remains more general-purpose; the graph is more reliable, but only for what it already knows to look for.

# Coding!: Extract Real Facts, Build a Real Schema, Ask a Real Multi-Hop Question

**A genuinely Different extraction prompt:** Not "is this a recurrring topic" (Day 1's question), today's extractor reads a real retrieved chunk and pulls out structured financial facts: which segment, which metric, which fiscal year, what value.

**A Schema That Represents the Document, Not the User:** `(company)-[:HAS_SEGMENT]->(Segment)-[:REPORTED]->(FinancialFact) {metric_type, fiscal_year, amount_millions})`, no `user` node anywhere in this part of the graph, because these facts belong to the document, not to any one person asking about it.

**One real, Deterministic Multi-Hop query:** A Single, Cypher query that traverses from the company node, through both segments, through both fiscal years, and returns all four members in one deterministic pass, then compute the two growth rates and the comparison in Python, from graph data, not from an LLM's read of retrieved text.

In [2]:
# Connecting to a real Graph database

from neo4j import GraphDatabase
import os

# AuraDB free tier: get these three values from your Neo4j Aura console
# after creating a free instance — no local install needed
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

with neo4j_driver.session() as session:
    result = session.run("RETURN 'connected' AS status")
    print(f"✅ Neo4j: {result.single()['status']}")
    
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext
from dataclasses import dataclass
from dotenv import load_dotenv, find_dotenv
from groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from typing import Optional

Settings.embed_model = HuggingFaceEmbedding(
    model_name='BAAI/bge-m3'
)

load_dotenv(find_dotenv())

client = Groq()

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# TOKEN USER TRACKER
@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()


# TOKEN aware LLM Caller

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

# RETRIEVAL WITH CONFIDENCE SCORING
RELEVANCE_THRESHOLD = 0.5

def retrieve_with_confidence(query: str) -> tuple[list, float]:
    """Returns retrieved docs and the top chunk's confidence score.
    Uses cosine similarity to score from your vectore store.
    """

    # creating a native llamaindex retriever from my initialized index
    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)

    # reults is a list of (document, score) tuples
    # lower score = more similar in FAISS  (L2 distance); invert if needed
    # For Cosine similarity stores, higher = better

    if not results:
        return [], 0.0
    
    
    top_score = float(results[0].score) if results[0].score is not None else 0.0

    print(f"📣 Top retrieval score : {top_score:.3f} (threshold: {RELEVANCE_THRESHOLD})")
    return results, top_score


✅ Neo4j: connected


c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# The Fact Extractor, Structured, Not "Is This Worth Tracking"

import re


FACT_EXTRACTOR_PROMPT = """You extract STRUCTURED FINANCIAL FACTS from a chunk of Apple's
FY2024 10-K. A fact has a segment name, a metric type, a fiscal year, and a dollar value.

Only extract facts EXPLICITLY stated as numbers in the text. Do not infer or calculate.

Respond in EXACTLY this format, one fact per line, or NONE if no clear fact is present:
FACT: <segment>|<metric_type>|<fiscal_year>|<amount_millions>

Example:
FACT: Services|revenue|FY2024|96169
FACT: iPhone|revenue|FY2023|200583
"""

def extract_financial_facts(chunk_text: str) -> list[dict]:
    raw = tracked_llm_call(
        messages=[{"role": "user", "content": f"Chunk:\n{chunk_text}"}],
        system=FACT_EXTRACTOR_PROMPT
    )
    facts = []
    for line in raw.split("\n"):
        match = re.match(r"FACT:\s*(.+?)\|(.+?)\|(.+?)\|(\d+)", line.strip())
        if match:
            facts.append({
                "segment": match.group(1).strip(),
                "metric_type": match.group(2).strip(),
                "fiscal_year": match.group(3).strip(),
                "amount_millions": int(match.group(4)),
            })
    return facts

In [4]:
# Building the Document Knowledge Graph

def write_fact_to_graph(fact: dict):
    with neo4j_driver.session() as session:
        session.run(
            """
            MERGE (c:Company {name: "Apple"})
            MERGE (s:Segment {name: $segment})
            MERGE (c)-[:HAS_SEGMENT]->(s)
            MERGE (f:FinancialFact {
                segment: $segment, metric_type: $metric_type, fiscal_year: $fiscal_year
            })
            SET f.amount_millions = $amount_millions
            MERGE (s)-[:REPORTED]->(f)
            """,
            **fact
        )
    print(f"  ✅ {fact['segment']} {fact['metric_type']} {fact['fiscal_year']}: "
          f"${fact['amount_millions']}M")


# Pull real chunks likely to contain segment revenue figures across both fiscal years
queries = [
    "Services segment net sales fiscal year 2024 fiscal year 2023",
    "iPhone net sales fiscal year 2024 fiscal year 2023",
]

all_facts = []
for q in queries:
    docs, score = retrieve_with_confidence(q)
    for doc in docs[:2]:
        chunk = doc.node.get_content() if hasattr(doc, "node") else doc.page_content
        facts = extract_financial_facts(chunk)
        all_facts.extend(facts)

print(f"\n📋 Extracted {len(all_facts)} structured facts")
for fact in all_facts:
    write_fact_to_graph(fact)


📣 Top retrieval score : 0.828 (threshold: 0.5)
📣 Top retrieval score : 0.880 (threshold: 0.5)

📋 Extracted 53 structured facts
  ✅ Americas Net sales FY2024: $167045M
  ✅ Americas Operating income FY2024: $67656M
  ✅ Americas Net sales FY2023: $162560M
  ✅ Americas Operating income FY2023: $60508M
  ✅ Americas Net sales FY2022: $169658M
  ✅ Americas Operating income FY2022: $62683M
  ✅ Europe Net sales FY2024: $101328M
  ✅ Europe Operating income FY2024: $41790M
  ✅ Europe Net sales FY2023: $94294M
  ✅ Europe Operating income FY2023: $36098M
  ✅ Europe Net sales FY2022: $95118M
  ✅ Europe Operating income FY2022: $35233M
  ✅ Greater China Net sales FY2024: $66952M
  ✅ Greater China Operating income FY2024: $27082M
  ✅ Greater China Net sales FY2023: $72559M
  ✅ Greater China Operating income FY2023: $30328M
  ✅ Greater China Net sales FY2022: $74200M
  ✅ iPhone revenue FY2024: $201183M
  ✅ iPhone revenue FY2023: $200583M
  ✅ iPhone revenue FY2022: $205489M
  ✅ Mac revenue FY2024: $2998

In [5]:
# The Real Multi-Hop Query, One Deterministic Traversal

import json

def compare_segment_growth(segment_a: str, segment_b: str, year_early: str, year_late: str) -> dict:
    with neo4j_driver.session() as session:
        result = session.run(
            """
            MATCH (sa:Segment {name: $segment_a})-[:REPORTED]->(fa_early:FinancialFact {fiscal_year: $year_early, metric_type: "revenue"})
            MATCH (sa)-[:REPORTED]->(fa_late:FinancialFact {fiscal_year: $year_late, metric_type: "revenue"})
            MATCH (sb:Segment {name: $segment_b})-[:REPORTED]->(fb_early:FinancialFact {fiscal_year: $year_early, metric_type: "revenue"})
            MATCH (sb)-[:REPORTED]->(fb_late:FinancialFact {fiscal_year: $year_late, metric_type: "revenue"})
            RETURN fa_early.amount_millions AS a_early, fa_late.amount_millions AS a_late,
                   fb_early.amount_millions AS b_early, fb_late.amount_millions AS b_late
            """,
            segment_a=segment_a, segment_b=segment_b, year_early=year_early, year_late=year_late
        )
        row = result.single()

    if not row:
        return {"found": False, "reason": "One or more required facts were never extracted into the graph"}

    growth_a = (row["a_late"] - row["a_early"]) / row["a_early"] * 100
    growth_b = (row["b_late"] - row["b_early"]) / row["b_early"] * 100
    return {
        "found": True,
        f"{segment_a}_growth_pct": round(growth_a, 2),
        f"{segment_b}_growth_pct": round(growth_b, 2),
        "faster_growing": segment_a if growth_a > growth_b else segment_b,
    }

result = compare_segment_growth("Services", "iPhone", "FY2023", "FY2024")
print(f"\n{'='*60}\nMULTI-HOP RESULT — deterministic, from the graph, no LLM reasoning involved\n{'='*60}")
print(json.dumps(result, indent=2))


MULTI-HOP RESULT — deterministic, from the graph, no LLM reasoning involved
{
  "found": true,
  "Services_growth_pct": 12.87,
  "iPhone_growth_pct": 0.3,
  "faster_growing": "Services"
}


In [6]:
# Contrast, What a Single vector Retrieval Gives You for the same question

comparison_question = ("Compare Apple's Services segment revenue growth rate to its iPhone "
                        "segment revenue growth rate from FY2023 to FY2024.")
docs, score = retrieve_with_confidence(comparison_question)
retrieved_text = "\n---\n".join([d.node.get_content()[:200] for d in docs])

print(f"\n{'='*60}\nSINGLE VECTOR RETRIEVAL — same question, one search, top {len(docs)} chunks\n{'='*60}")
print(retrieved_text)
print(f"\nCheck by hand: do all FOUR needed numbers (Services FY23/FY24, iPhone FY23/FY24)")
print(f"actually appear in this retrieval, or did the single search only surface some of them?")

📣 Top retrieval score : 0.857 (threshold: 0.5)

SINGLE VECTOR RETRIEVAL — same question, one search, top 4 chunks

Products and Services Performance

The following table shows net sales by category for 2024, 2023 and 2022 (dollars in millions):

|                                 | 2024      | Change | 2023      |
---
# Net sales disaggregated by significant products and services for 2024, 2023 and 2022 were as follows (in millions):

|                                 | 2024          | 2023          | 2022         
---
# Net sales disaggregated by significant products and services for 2024, 2023 and 2022 were as follows (in millions):

|                                 | 2024          | 2023          | 2022         
---
Products and Services Performance

The following table shows net sales by category for 2024, 2023 and 2022 (dollars in millions):

|                                 | 2024      | Change | 2023      | 

Check by hand: do all FOUR needed numbers (Services FY23/FY24, iPhone